In [ ]:
import os

# Montar Google Drive (solo en Colab)
from google.colab import drive
drive.mount('/content/drive')

# Ruta base del dataset (ajústala si es necesario)
base_path = "/content/drive/MyDrive/PASD BLOG/RDD2022_clean"

# Lista de carpetas (países)
countries = [
    "China_MotorBike",
    "Czech",
    "India",
    "Japan",
    "Norway",
    "United_States"
]

total_images = 0

for country in countries:
    images_path = os.path.join(base_path, country, "train", "images")

    if os.path.exists(images_path):
        # Contar solo archivos (evita carpetas ocultas)
        num_images = len([
            f for f in os.listdir(images_path)
            if os.path.isfile(os.path.join(images_path, f))
        ])

        print(f"{country}: {num_images} imágenes")
        total_images += num_images
    else:
        print(f"{country}: ruta no encontrada")

print("\nTotal de imágenes en el dataset:", total_images)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
China_MotorBike: 740 imágenes
Czech: 708 imágenes
India: 855 imágenes
Japan: 2599 imágenes
Norway: 1983 imágenes
United_States: 1912 imágenes

Total de imágenes en el dataset: 8797


In [ ]:
import json
from collections import Counter

# Ruta del metadata (ajústala si es necesario)
metadata_path = "/content/drive/MyDrive/PASD BLOG/features/dinov2/train_metadata.json"

with open(metadata_path, encoding="utf-8") as f:
    meta = json.load(f)

CLASSES    = ["D00", "D10", "D20"]
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15

# ── Conteo ────────────────────────────────────────────
class_counter   = Counter()
country_counter = Counter()

for sample in meta:
    classes_en_imagen = set(sample.get("classes", []))
    country_counter[sample["country"]] += 1
    for cls in classes_en_imagen:
        if cls in CLASSES:
            class_counter[cls] += 1

n_total = len(meta)
n_val   = int(n_total * VAL_SPLIT)
n_test  = int(n_total * TEST_SPLIT)
n_train = n_total - n_val - n_test

# ── Resultados ────────────────────────────────────────
print(f"Total imágenes: {n_total}")

print(f"\nSplit 70/15/15:")
print(f"  Train:      {n_train} imágenes")
print(f"  Validación: {n_val} imágenes")
print(f"  Test:       {n_test} imágenes")

print(f"\nDistribución por clase:")
for cls in CLASSES:
    count         = class_counter[cls]
    pct           = count / n_total * 100
    expected_val  = int(count * VAL_SPLIT)
    expected_test = int(count * TEST_SPLIT)
    viable_val    = "✅" if expected_val  >= 20 else "⚠️  POCO"
    viable_test   = "✅" if expected_test >= 20 else "⚠️  POCO"
    print(f"  {cls}: {count:>5} imágenes ({pct:.1f}%)"
          f" → val≈{expected_val} {viable_val} | test≈{expected_test} {viable_test}")

print(f"\nDistribución por país:")
for country, count in country_counter.most_common():
    print(f"  {country}: {count} imágenes")

Total imágenes: 8797

Split 70/15/15:
  Train:      6159 imágenes
  Validación: 1319 imágenes
  Test:       1319 imágenes

Distribución por clase:
  D00:  5852 imágenes (66.5%) → val≈877 ✅ | test≈877 ✅
  D10:  2968 imágenes (33.7%) → val≈445 ✅ | test≈445 ✅
  D20:  3058 imágenes (34.8%) → val≈458 ✅ | test≈458 ✅

Distribución por país:
  Japan: 2599 imágenes
  Norway: 1983 imágenes
  United_States: 1912 imágenes
  India: 855 imágenes
  China_MotorBike: 740 imágenes
  Czech: 708 imágenes
